In [19]:
import ollama

MODEL = "gemma4:latest"

In [2]:
# PART 1: BASIC OLLAMA CONNECTION
# This was the first version of DevMentor.
# It accepted a dynamic user question, sent it to Gemma through Ollama,
# and returned the model's response.

# def ask_llm(question):
#     response = ollama.chat(
#         model=MODEL,
#         messages=[
#             {
#                 "role": "user",
#                 "content": question
#             }
#         ]
#     )
#
#     return response["message"]["content"]
#
#
# question = input("You: ")
# answer = ask_llm(question)
#
# print("\nDevMentor:")
# print(answer)

In [3]:
# PART 2: SYSTEM PROMPT ENGINEERING
# A system prompt was introduced to give DevMentor a clear role,
# target audience, explanation style, and behavioural rules.

SYSTEM_PROMPT = """
You are DevMentor, a local programming assistant for junior developers.

Your job is to explain programming concepts clearly and practically.

Rules:
- Use simple and direct language.
- Introduce technical terms before using them heavily.
- Give a short explanation first.
- Use one practical example or analogy when useful.
- Show code only when it helps.
- Keep most answers concise unless the user asks for more detail.
- Avoid unnecessary complexity.
- If you are unsure, say so clearly instead of guessing.
- Keep answers focused on programming and software development.
""".strip()

In [4]:
# PART 2: UPDATED FUNCTION WITH SYSTEM PROMPT
# This version added the system prompt to the messages sent to the model.
# It did not yet maintain conversation history.

# def ask_llm(question):
#     response = ollama.chat(
#         model=MODEL,
#         messages=[
#             {
#                 "role": "system",
#                 "content": SYSTEM_PROMPT
#             },
#             {
#                 "role": "user",
#                 "content": question
#             }
#         ]
#     )
#
#     return response["message"]["content"]
#
#
# question = input("You: ")
# answer = ask_llm(question)
#
# print("\nDevMentor:")
# print(answer)

In [5]:
# PART 3: CONVERSATION MEMORY
# A messages list was introduced to store the system prompt
# and the ongoing conversation between the user and DevMentor.

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
# # PART 3: UPDATED FUNCTION WITH CONVERSATION MEMORY
# # The user's message is added to the conversation history.
# # The full history is sent to the model.
# # The assistant's response is then stored for the next turn.

# def ask_llm(question):
#     messages.append(
#         {
#             "role": "user",
#             "content": question
#         }
#     )

#     response = ollama.chat(
#         model=MODEL,
#         messages=messages
#     )

#     answer = response["message"]["content"]

#     messages.append(
#         {
#             "role": "assistant",
#             "content": answer
#         }
#     )

#     return answer

In [7]:
# PART 3: MEMORY TEST
# This was used to test whether DevMentor could understand
# follow-up questions without the topic being repeated.

# question = input("You: ")
# answer = ask_llm(question)
#
# print("\nDevMentor:")
# print(answer)

In [8]:
# PART 4: CONVERSATION CONTROLS
# This function clears previous user and assistant messages
# while preserving the DevMentor system prompt.

def reset_conversation():
    global messages

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    print("Conversation reset.")

In [9]:
# PART 4: DISPLAY CONVERSATION HISTORY
# This displays the user and assistant messages.
# The system prompt is excluded from the visible history.

def display_history():
    conversation = messages[1:]

    if not conversation:
        print("No conversation history.")
        return

    for number, message in enumerate(conversation, start=1):
        role = message["role"].upper()
        content = message["content"]

        print(f"{number}. {role}: {content}\n")

In [ ]:
# # PART 4: INTERACTIVE CHAT LOOP
# # The application continues running until the user exits.
# # It also supports /history, /reset, and /exit commands.

# while True:
#     question = input("You: ").strip()

#     if question == "/exit":
#         print("Goodbye!")
#         break

#     elif question == "/reset":
#         reset_conversation()
#         continue

#     elif question == "/history":
#         display_history()
#         continue
#     answer = ask_llm(question)

#     print("\nDevMentor:")
#     print(answer)
#     print()


DevMentor:
Goodbye! 👋

I'll be here whenever you need another coding session. Happy learning, and keep practicing those concepts!

Goodbye!


In [13]:
# PART 5: ERROR HANDLING
# This version improves the Ollama function so common errors
# are handled with clear messages instead of raw Python tracebacks.

def ask_llm(question):
    messages.append(
        {
            "role": "user",
            "content": question
        }
    )

    try:
        response = ollama.chat(
            model=MODEL,
            messages=messages
        )

        answer = response["message"]["content"]

        messages.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        return answer

    except Exception as error:
        # Remove the user message because the request failed.
        messages.pop()

        return f"Unable to get a response from Ollama. Error: {error}"

In [17]:
# PART 5: UPDATED INTERACTIVE CHAT LOOP
# This version prevents blank input from being sent to the model.
# It keeps the existing /history, /reset, and /exit commands.

while True:
    question = input("You: ").strip()

    # Prevent empty input from being sent to Ollama.
    if not question:
        print("Please enter a question.")
        continue

    if question == "/exit":
        print("Goodbye!")
        break

    elif question == "/reset":
        reset_conversation()
        continue

    elif question == "/history":
        display_history()
        continue

    answer = ask_llm(question)

    print("\nDevMentor:")
    print(answer)
    print()


DevMentor:
Unable to get a response from Ollama. Error: model 'wrong-model' not found (status code: 404)

Goodbye!


In [ ]:
# PART 7: PROMPT ENGINEERING EXPERIMENT
# Three different system prompts are tested with the same questions.
# The goal is to observe how prompt detail and constraints affect model behaviour.

In [20]:
PROMPT_A = """
You are a programming assistant.
""".strip()


PROMPT_B = """
You are DevMentor, a programming tutor for junior developers.

Explain programming concepts clearly and practically.
Use simple language.
Introduce technical terms before using them heavily.
Use practical examples where useful.
If you are unsure, say so clearly.
""".strip()


PROMPT_C = """
You are DevMentor, a programming tutor for junior developers.

Rules:
- Explain the concept before showing code.
- Keep the introduction short.
- Use simple and direct language.
- Use one practical example when useful.
- Avoid unnecessary detail.
- Do not guess when unsure.
- Keep the answer focused on programming.
""".strip()

In [21]:
def test_prompt(system_prompt, question):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response["message"]["content"]

In [22]:
questions = [
    "Explain REST APIs.",
    "Explain recursion.",
    "What is dependency injection?",
    "Show me an example."
]

In [23]:
prompts = {
    "Prompt A - Minimal": PROMPT_A,
    "Prompt B - Detailed": PROMPT_B,
    "Prompt C - Constrained": PROMPT_C
}

for prompt_name, system_prompt in prompts.items():

    print("=" * 70)
    print(prompt_name)
    print("=" * 70)

    for question in questions:

        print(f"\nQuestion: {question}")

        answer = test_prompt(
            system_prompt,
            question
        )

        print(f"\nAnswer:\n{answer}")
        print("-" * 70)

Prompt A - Minimal

Question: Explain REST APIs.

Answer:
This is a comprehensive explanation of REST APIs, designed to cover the theory, the practical implementation, and the core principles.

---

# 📘 What Are REST APIs?

In simple terms, a **REST API** is an architectural style—a set of rules and best practices—used for designing web services. It defines *how* different computer systems (the client and the server) should communicate with each other over standard web protocols, primarily HTTP.

If you think of the internet as a giant postal system, an API is the specific standardized set of addresses and formats for sending letters (data) back and forth. REST is the *rulebook* that makes that postal system efficient, predictable, and easy for any system to use.

***

## 🧱 Part 1: The Basics (APIs vs. REST)

It's crucial to understand the difference between an API and REST:

1.  **API (Application Programming Interface):** This is the *contract*. It is the defined set of rules and met

In [ ]:
# PART 8: MEMORY INVESTIGATION
# This experiment checks whether the model actually remembers information,
# or whether it only knows what is included in the conversation history.

In [24]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [25]:
print(ask_llm("My favorite programming language is Python."))

That's a great choice! Python is known for its readability and simplicity, which makes it fantastic for beginners but powerful enough for professional projects.

It's often praised because its structure is very close to plain English, allowing you to focus more on solving the problem than on wrestling with complicated syntax.

Since you're comfortable with Python, what area are you currently working on?

Are you interested in:
1. **Web Development** (using frameworks like Django or Flask)?
2. **Data Science** (using libraries like Pandas or NumPy)?
3. **Automation** (writing small scripts to manage files)?

Let me know what you'd like to learn or review next!


In [26]:
print(ask_llm("Explain interfaces."))

That's a slightly abstract topic, but it's one of the most critical concepts in Object-Oriented Programming (OOP). I'll break it down for you using simple terms.

### 🔑 What is an Interface? (The Concept)

Think of an interface as a **contract** or a **guaranteed blueprint**.

When a programmer uses an interface, they aren't caring *how* something works; they only care that it *can* do certain things.

*   **Definition:** An interface specifies a set of methods (functions) that any class must implement if it claims to adhere to that interface.
*   **The Promise:** It says, "Any class that uses this contract *must* have a `save()` method and a `load()` method." It doesn't care what those methods do internally—only that they exist and have the correct structure.

---

### 🚗 Analogy: The Remote Control

Imagine a TV remote control (the Interface).

The remote promises certain actions:
1. `Power On`
2. `Change Channel`
3. `Adjust Volume`

*   **The Implementation:** Your TV (the Class) is 

In [27]:
print(ask_llm("What is my favorite programming language?"))

Based on our conversation, your favorite programming language is **Python**! 😊


In [28]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": "Explain interfaces."
    }
]

In [29]:
print(ask_llm("What is my favorite programming language?"))

I don't have access to your personal information or preferences, so I don't know what your favorite programming language is! 😊

However, if you tell me what languages you like, I can help you understand their strengths, weaknesses, and when it's best to use them.

Are you working on any projects right now? We can talk about the strengths of different languages based on your goals!
